In [ ]:
import os

os.environ['KAGGLE_API_TOKEN'] = "kaggle API"

In [2]:
!kaggle competitions download -c super-ai-engineer-ss-6-word-segmentation

!ls

!unzip super-ai-engineer-ss-6-word-segmentation.zip -d super-ai-engineer-ss-6-word-segmentation

!ls super-ai-engineer-ss-6-word-segmentation

100% 1.95M/1.95M [00:00<00:00, 205MB/s]

sample_data  super-ai-engineer-ss-6-word-segmentation.zip
Archive:  super-ai-engineer-ss-6-word-segmentation.zip
  inflating: super-ai-engineer-ss-6-word-segmentation/LST20 Annotation Guideline.pdf  
  inflating: super-ai-engineer-ss-6-word-segmentation/LST20 Brief Specification.pdf  
  inflating: super-ai-engineer-ss-6-word-segmentation/ws_list.txt  
  inflating: super-ai-engineer-ss-6-word-segmentation/ws_sample_submission.csv  
  inflating: super-ai-engineer-ss-6-word-segmentation/ws_test.txt  
'LST20 Annotation Guideline.pdf'   ws_list.txt		      ws_test.txt
'LST20 Brief Specification.pdf'    ws_sample_submission.csv


In [3]:
!pip install -U transformers accelerate datasets sentencepiece pythainlp

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 113.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 53.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.8/19.8 MB 97.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 48.5 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [4]:
!pip install datasets==2.19.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 22.9 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.8.4
    Uninstalling datasets-4.8.4:
      Successfully uninstalled datasets-4.8.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.3.1 which is incompatible.


In [5]:
import os

!pip install -q gdown
!gdown "1uCjU13JqXxuUsErE3SPTB7VxL1pfNqec" -O LST20.tar.gz
!tar -xzf LST20.tar.gz

extracted_dir = "LST20_Corpus"
if not os.path.exists(extracted_dir):
    extracted_dir = "LST20Corpus"

from datasets import load_dataset
dataset = load_dataset("lst-nectec/lst20", data_dir=extracted_dir, trust_remote_code=True)

Downloading...
From: https://drive.google.com/uc?id=1uCjU13JqXxuUsErE3SPTB7VxL1pfNqec
To: /content/LST20.tar.gz
100% 13.6M/13.6M [00:00<00:00, 60.3MB/s]


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/63310 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5620 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5250 [00:00<?, ? examples/s]

#3/4

In [ ]:
!pip install -q transformers[torch] accelerate evaluate datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.3 MB/s eta 0:00:00


In [ ]:
!pip install -q transformers[torch] accelerate datasets sentencepiece

In [ ]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["TORCH_USE_CUDA_DSA"] = "1"

import torch
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(False)
torch.backends.cuda.enable_math_sdp(True)

import pandas as pd
from transformers import AutoTokenizer, AutoModelForTokenClassification, CamembertTokenizer, TrainingArguments, Trainer, DataCollatorForTokenClassification
from datasets import concatenate_datasets
from tqdm import tqdm

model_name = "clicknext/phayathaibert"
label_list = ["O", "B_WORD", "I_WORD"]
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for i, l in enumerate(label_list)}

tokenizer = CamembertTokenizer.from_pretrained(model_name, legacy=True)
vocab_size = len(tokenizer)

model = AutoModelForTokenClassification.from_pretrained(
    model_name, num_labels=3, id2label=id2label, label2id=label2id,
    ignore_mismatched_sizes=True, attn_implementation="eager"
)
model.resize_token_embeddings(vocab_size)

def prepare_data(examples):
    all_texts = []
    all_char_labels = []

    for tokens in examples['tokens']:
        text = ""
        tags = []
        for word in tokens:
            if word == '_' or word.isspace():
                text += word
                tags.extend([0] * len(word))
            else:
                text += word
                tags.append(1)
                tags.extend([2] * (len(word) - 1))
        all_texts.append(text)
        all_char_labels.append(tags)

    tokenized = tokenizer(all_texts, truncation=True, padding='max_length', max_length=256, return_offsets_mapping=True)

    safe_input_ids = []
    for seq in tokenized["input_ids"]:
        safe_input_ids.append([max(0, min(idx, vocab_size - 1)) for idx in seq])
    tokenized["input_ids"] = safe_input_ids

    labels = []
    for i, mapping in enumerate(tokenized["offset_mapping"]):
        doc_labels = []
        char_tags = all_char_labels[i]
        for start, end in mapping:
            if start == end:
                doc_labels.append(-100)
            else:
                idx = min(start, len(char_tags) - 1)
                doc_labels.append(char_tags[idx])
        labels.append(doc_labels)

    tokenized["labels"] = labels
    tokenized.pop("offset_mapping")
    return tokenized

combined_ds = concatenate_datasets([dataset['train'], dataset['validation'], dataset['test']])
train_encoded = combined_ds.map(prepare_data, batched=True, remove_columns=combined_ds.column_names)

training_args = TrainingArguments(
    output_dir='./results_master',
    num_train_epochs=3,
    per_device_train_batch_size=64,
    bf16=True,
    learning_rate=3e-5,
    eval_strategy="no",
    save_strategy="no",
    logging_steps=100,
    dataloader_num_workers=2,
    report_to="none"
)

trainer = Trainer(
    model=model, args=training_args, train_dataset=train_encoded,
    data_collator=DataCollatorForTokenClassification(tokenizer)
)

trainer.train()

tokenizer_config.json:   0%|          | 0.00/527 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.26M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.4M [00:00<?, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/364 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

CamembertForTokenClassification LOAD REPORT from: clicknext/phayathaibert
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.weight               | MISSING    | 
classifier.bias                 | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/74180 [00:00<?, ? examples/s]

Step,Training Loss
100,0.142820
200,0.049963
300,0.044321
400,0.038931
500,0.036905
600,0.037318
700,0.035226
800,0.032399
900,0.032621
1000,0.031296


TrainOutput(global_step=3480, training_loss=0.031173413957672556, metrics={'train_runtime': 727.7862, 'train_samples_per_second': 305.777, 'train_steps_per_second': 4.782, 'total_flos': 2.907475298270208e+16, 'train_loss': 0.031173413957672556, 'epoch': 3.0})

In [ ]:
epochs = 10
model.train()

for epoch in range(epochs):
    total_loss, correct, total = 0, 0, 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
    for inputs, labels in pbar:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)

        loss = criterion(outputs.view(-1, len(tag2idx)), labels.view(-1))
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = torch.argmax(outputs, dim=2)
        mask = labels != 0
        correct += (preds[mask] == labels[mask]).sum().item()
        total += mask.sum().item()
        pbar.set_postfix({'Loss': f"{loss.item():.4f}", 'Acc': f"{correct/total:.4f}"})

Epoch 10/10: 100%|██████████| 184/184 [00:16<00:00, 10.88it/s, Loss=0.0287, Acc=0.9912]


In [ ]:
base_path = 'super-ai-engineer-ss-6-word-segmentation'
with open(f'{base_path}/ws_test.txt', 'r', encoding='utf-8') as f:
    test_text = f.read()

model.eval()
test_chars = list(test_text)
test_idx = [char2idx.get(c, char2idx['<UNK>']) for c in test_chars]
final_preds = []

with torch.no_grad():
    for i in tqdm(range(0, len(test_idx), SEQ_LEN)):
        chunk = test_idx[i : i + SEQ_LEN]
        pad_len = SEQ_LEN - len(chunk) if len(chunk) < SEQ_LEN else 0
        chunk += [char2idx['<PAD_O>']] * pad_len

        tensor_chunk = torch.tensor([chunk], dtype=torch.long).to(device)
        output = model(tensor_chunk)
        preds = torch.argmax(output, dim=2).squeeze().cpu().numpy()

        actual_len = SEQ_LEN if len(test_idx[i:]) >= SEQ_LEN else len(test_idx[i:])
        final_preds.extend(preds[:actual_len])

submission_tags = []
prev_tag = 'O'

for char, tag_idx in zip(test_chars, final_preds):
    if not char.isspace():
        tag_name = idx2tag[tag_idx]

        if prev_tag in ['E_WORD', 'O'] and tag_name in ['I_WORD', 'E_WORD']:
            tag_name = 'B_WORD'
        if tag_name == '<PAD_O>':
            tag_name = 'B_WORD'

        submission_tags.append(tag_name)
        prev_tag = tag_name
    else:
        prev_tag = 'O'

sample_sub = pd.read_csv(f'{base_path}/ws_sample_submission.csv')
if len(submission_tags) == len(sample_sub):
    sample_sub['Predicted'] = submission_tags
    sample_sub.to_csv('ws_bilstm_trained_full.csv', index=False)
    print("\n✅")
else:
    print("\n🚨")

100%|██████████| 146/146 [00:01<00:00, 88.58it/s]


✅


#4/4

In [7]:
!pip install pytorch-crf -q

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchcrf import CRF
from tqdm import tqdm
import pandas as pd
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


tag2idx = {'<PAD>': 0, 'B_WORD': 1, 'I_WORD': 2, 'E_WORD': 3, 'O': 4}
idx2tag = {v: k for k, v in tag2idx.items()}

def extract_char_tags(tokens):
    chars, tags = [], []
    for word in tokens:
        if word == '<_>' or word.isspace():
            chars.append(' ')
            tags.append('O')
        else:
            if len(word) == 1:
                chars.append(word[0])
                tags.append('B_WORD')
            else:
                for i, c in enumerate(word):
                    chars.append(c)
                    if i == 0: tags.append('B_WORD')
                    elif i == len(word) - 1: tags.append('E_WORD')
                    else: tags.append('I_WORD')
    return chars, tags

train_chars_list, train_tags_list = [], []
all_chars = set()
for item in tqdm(dataset['train']):
    chars, tags = extract_char_tags(item['tokens'])
    train_chars_list.append(chars)
    train_tags_list.append(tags)
    all_chars.update(chars)

char2idx = {char: idx + 2 for idx, char in enumerate(all_chars)}
char2idx['<PAD>'] = 0
char2idx['<UNK>'] = 1

def collate_fn(batch):
    max_len = min(max([len(b[0]) for b in batch]), 512)
    batch_chars, batch_tags, batch_mask = [], [], []

    for chars, tags in batch:
        c_ids = [char2idx.get(c, char2idx['<UNK>']) for c in chars[:max_len]]
        t_ids = [tag2idx[t] for t in tags[:max_len]]

        pad_len = max_len - len(c_ids)
        batch_chars.append(c_ids + [char2idx['<PAD>']] * pad_len)
        batch_tags.append(t_ids + [tag2idx['<PAD>']] * pad_len)
        batch_mask.append([1]*len(c_ids) + [0]*pad_len)

    return torch.tensor(batch_chars), torch.tensor(batch_tags), torch.tensor(batch_mask, dtype=torch.uint8)

train_dataset = list(zip(train_chars_list, train_tags_list))
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, collate_fn=collate_fn)

class BiLSTM_CRF(nn.Module):
    def __init__(self, vocab_size, tagset_size, embed_dim=128, hidden_dim=256):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim // 2, num_layers=2,
                            bidirectional=True, batch_first=True, dropout=0.3)
        self.hidden2tag = nn.Linear(hidden_dim, tagset_size)
        self.crf = CRF(tagset_size, batch_first=True)

    def forward(self, x, mask):
        embs = self.embedding(x)
        lstm_out, _ = self.lstm(embs)
        emissions = self.hidden2tag(lstm_out)
        return emissions

    def loss(self, x, tags, mask):
        emissions = self.forward(x, mask)
        return -self.crf(emissions, tags, mask=mask.bool(), reduction='mean')

    def decode(self, x, mask):
        emissions = self.forward(x, mask)
        return self.crf.decode(emissions, mask=mask.bool())

model = BiLSTM_CRF(len(char2idx), len(tag2idx)).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.003)

model.train()
for epoch in range(7):
    total_loss = 0
    for chars, tags, mask in tqdm(train_loader, desc=f"Epoch {epoch+1}/7"):
        chars, tags, mask = chars.to(device), tags.to(device), mask.to(device)
        optimizer.zero_grad()
        loss = model.loss(chars, tags, mask)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"📉 Epoch {epoch+1} Loss: {total_loss/len(train_loader):.4f}")

test_file = 'super-ai-engineer-ss-6-word-segmentation/ws_test.txt'
with open(test_file, 'r', encoding='utf-8') as f:
    test_text = f.read()

model.eval()
all_preds = []
current_chunk_chars = []

with torch.no_grad():
    for char in test_text:
        current_chunk_chars.append(char)
        if char == '\n':
            c_ids = [char2idx.get(c, char2idx['<UNK>']) for c in current_chunk_chars]
            c_tensor = torch.tensor([c_ids]).to(device)
            m_tensor = torch.tensor([[1]*len(c_ids)], dtype=torch.uint8).to(device)

            preds = model.decode(c_tensor, m_tensor)[0]
            all_preds.extend(preds)
            current_chunk_chars = []

    if len(current_chunk_chars) > 0:
        c_ids = [char2idx.get(c, char2idx['<UNK>']) for c in current_chunk_chars]
        c_tensor = torch.tensor([c_ids]).to(device)
        m_tensor = torch.tensor([[1]*len(c_ids)], dtype=torch.uint8).to(device)

        preds = model.decode(c_tensor, m_tensor)[0]
        all_preds.extend(preds)

assert len(all_preds) == len(test_text), "จำนวนผลทำนายไม่เท่ากับจำนวนอักษร"

sample_sub_path = 'super-ai-engineer-ss-6-word-segmentation/ws_sample_submission.csv'
sample_sub = pd.read_csv(sample_sub_path)

final_tags = []
for id_val in sample_sub['Id']:
    char_idx = id_val - 1
    pred = all_preds[char_idx]
    tag_str = idx2tag.get(pred, 'O')

    if tag_str not in ['B_WORD', 'I_WORD', 'E_WORD']:
        tag_str = 'B_WORD'

    final_tags.append(tag_str)

sample_sub['Predicted'] = final_tags
submission_file = 'submission_LST20_CRF_MASTER++.csv'
sample_sub.to_csv(submission_file, index=False)

Epoch 1/7: 100%|██████████| 495/495 [04:52<00:00,  1.69it/s]


📉 Epoch 1 Loss: 19.6150


Epoch 2/7: 100%|██████████| 495/495 [04:52<00:00,  1.69it/s]


📉 Epoch 2 Loss: 5.7537


Epoch 3/7: 100%|██████████| 495/495 [04:51<00:00,  1.70it/s]


...

📉 Epoch 5 Loss: 2.7524


Epoch 6/7: 100%|██████████| 495/495 [04:52<00:00,  1.69it/s]


📉 Epoch 6 Loss: 2.4642


Epoch 7/7: 100%|██████████| 495/495 [04:51<00:00,  1.70it/s]


📉 Epoch 7 Loss: 2.2534


In [20]:
!pip install pythainlp attacut deepcut pandas tqdm -q

import pandas as pd
from pythainlp.tokenize import word_tokenize
from pythainlp.util import dict_trie
from pythainlp.corpus.common import thai_words
from tqdm import tqdm
from collections import Counter
import torch

test_file = 'super-ai-engineer-ss-6-word-segmentation/ws_test.txt'
sample_sub_path = 'super-ai-engineer-ss-6-word-segmentation/ws_sample_submission.csv'

with open(test_file, 'r', encoding='utf-8') as f:
    test_text = f.read()

sample_sub = pd.read_csv(sample_sub_path)
valid_ids = set(sample_sub['Id'])
TEXT_LEN = len(test_text)

model.eval()
all_preds_bilstm = []
current_chunk = []

with torch.no_grad():
    for char in test_text:
        current_chunk.append(char)
        if (char.isspace() or char == '\n' or char == '\u200b') and len(current_chunk) > 200:
            c_ids = [char2idx.get(c, char2idx['<UNK>']) for c in current_chunk]
            c_tensor = torch.tensor([c_ids]).to(device)
            m_tensor = torch.tensor([[1]*len(c_ids)], dtype=torch.uint8).to(device)
            preds = model.decode(c_tensor, m_tensor)[0]
            all_preds_bilstm.extend(preds)
            current_chunk = []

    if len(current_chunk) > 0:
        c_ids = [char2idx.get(c, char2idx['<UNK>']) for c in current_chunk]
        c_tensor = torch.tensor([c_ids]).to(device)
        m_tensor = torch.tensor([[1]*len(c_ids)], dtype=torch.uint8).to(device)
        preds = model.decode(c_tensor, m_tensor)[0]
        all_preds_bilstm.extend(preds)

def get_char_tags(words_list, expected_len):
    tags = []
    for word in words_list:
        is_special = word.isspace() or word == '\n' or word == '\u200b'
        for i, _ in enumerate(word):
            if is_special:
                tags.append('B_WORD')
            else:
                if len(word) == 1: tags.append('B_WORD')
                elif i == 0: tags.append('B_WORD')
                elif i == len(word) - 1: tags.append('E_WORD')
                else: tags.append('I_WORD')
    if len(tags) < expected_len:
        tags.extend(['B_WORD'] * (expected_len - len(tags)))
    return tags[:expected_len]

words_deepcut = word_tokenize(test_text, engine='deepcut')
tags_deepcut = get_char_tags(words_deepcut, TEXT_LEN)

custom_words = set(thai_words())
try:
    for item in dataset['train']:
        for word in item['tokens']:
            if word != '<_>' and not word.isspace(): custom_words.add(word)
except: pass
custom_trie = dict_trie(dict_source=custom_words)
words_newmm = word_tokenize(test_text, custom_dict=custom_trie, engine='newmm')
tags_newmm = get_char_tags(words_newmm, TEXT_LEN)

results = []
current_id = 1

for i, char in enumerate(tqdm(test_text, desc="Final Polling")):
    if current_id in valid_ids:
        if char.isspace() or char == '\n' or char == '\u200b':
            results.append({'Id': current_id, 'Predicted': 'B_WORD'})
        else:
            p_bilstm = idx2tag.get(all_preds_bilstm[i], 'B_WORD')
            if p_bilstm not in ['B_WORD', 'I_WORD', 'E_WORD']: p_bilstm = 'B_WORD'

            p_deepcut = tags_deepcut[i]
            p_newmm = tags_newmm[i]

            if p_deepcut == p_bilstm or p_deepcut == p_newmm:
                final_tag = p_deepcut
            elif p_bilstm == p_newmm:
                final_tag = p_bilstm
            else:
                final_tag = p_deepcut

            results.append({'Id': current_id, 'Predicted': final_tag})
    current_id += 1

submission_df = pd.DataFrame(results)
final_sub = pd.merge(sample_sub[['Id']], submission_df, on='Id', how='left')
final_sub['Predicted'] = final_sub['Predicted'].fillna('B_WORD')
final_sub.to_csv('submission_lol.csv', index=False)

1164/1164 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step


Final Polling: 100%|██████████| 37248/37248 [00:00<00:00, 946724.57it/s]
